In [1]:
range_fn = "data/n4/silversword.n4.range.nex"
tree_fn = "data/n4/silversword.tre"
out_fn = "output/epoch"

   Missing Variable:	Variable os does not exist



In [2]:
geo_fn = "data/n4/hawaii.n4"
times_fn = geo_fn + ".times.txt"
dist_fn = geo_fn + ".distances.txt"

In [3]:
moves = VectorMoves()
monitors = VectorMonitors()

In [4]:
range_fn

   data/n4/silversword.n4.range.nex


In [5]:
dat_range_01 = readDiscreteCharacterData(range_fn)
n_areas <- dat_range_01.nchar()

   Successfully read one character matrix from file 'data/n4/silversword.n4.range.nex'


In [9]:
max_areas <- 2
n_states <- 0
for (k in 0:max_areas) n_states += choose(n_areas, k)

In [10]:
dat_range_n = formatDiscreteCharacterData(dat_range_01, "DEC", n_states)

In [12]:
state_desc = dat_range_n.getStateDescriptions()
state_desc_str = "state,range\n"
for (i in 1:state_desc.size()){
    state_desc_str += (i-1) + "," + state_desc[i] + "\n"
}
write(state_desc_str, file=out_fn+".state_labels.txt")

In [13]:
tree <- readTrees(tree_fn)[1]

   Attempting to read the contents of file "silversword.tre"
   Successfully read file


In [15]:
time_bounds <- readDataDelimitedFile(file=times_fn, delimiter=" ")
n_epochs <- time_bounds.size()

In [16]:
for (i in 1:n_epochs) {
      epoch_fn[i] = geo_fn + ".connectivity." + i + ".txt"
      connectivity[i] <- readDataDelimitedFile(file=epoch_fn[i], delimiter=" ")
}

In [17]:
distances <- readDataDelimitedFile(file=dist_fn, delimiter=" ")

In [18]:
rate_bg ~ dnLoguniform(1E-4,1E2)
rate_bg.setValue(1E-2)

In [19]:
moves.append( mvSlide(rate_bg, weight=4) )

In [20]:
dispersal_rate <- 1.0

In [21]:
distance_scale ~ dnUnif(0,20)
distance_scale.setValue(0.01)
moves.append( mvScale(distance_scale, weight=3) )

In [22]:
for (i in 1:n_epochs) {
    for (j in 1:n_areas) {
        for (k in 1:n_areas) {
            dr[i][j][k] <- 0.0
                if (connectivity[i][j][k] > 0) {
                    dr[i][j][k]  := dispersal_rate * exp(-distance_scale * distances[j][k])
                }
        }
    }
}

In [23]:
log_sd <- 0.5
log_mean <- ln(1) - 0.5*log_sd^2
extirpation_rate ~ dnLognormal(mean=log_mean, sd=log_sd)
moves.append( mvScale(extirpation_rate, weight=2) )

In [24]:
for (i in 1:n_epochs) {
    for (j in 1:n_areas) {
        for (k in 1:n_areas) {
            er[i][j][k] <- 0.0
        }
        er[i][j][j] := extirpation_rate
    }
}

In [25]:
for (i in 1:n_epochs) {
    Q_DEC[i] := fnDECRateMatrix(dispersalRates=dr[i],
                extirpationRates=er[i],
                maxRangeSize=max_areas)
}

In [26]:
for (i in 1:n_epochs) {
    time_max[i] <- time_bounds[i][1]
    time_min[i] <- time_bounds[i][2]
    if (i != n_epochs) {
        epoch_times[i] ~ dnUniform(time_min[i], time_max[i])
        moves.append( mvSlide(epoch_times[i], delta=(time_max[i]-time_min[i])/2) )
    } else {
        epoch_times[i] <- 0.0
    }
}

In [27]:
Q_DEC_epoch := fnEpoch(Q=Q_DEC, times=epoch_times, rates=rep(1,n_epochs))

In [45]:
epoch_times

   [ 3.100, 1.686, 0.393, 0.000 ]


In [43]:
time_bounds

   [ [ 3.7000, 2.2000 ] ,
     1.8000, 1.3000 ] ,
     0.7000, 0.3000 ] ,
     0.0000, 0.0000 ] ]


In [28]:
clado_event_types <- [ "s", "a" ]
p_sympatry ~ dnUniform(0,1)
p_allopatry := abs(1.0 - p_sympatry)
clado_type_probs := simplex(p_sympatry, p_allopatry)
moves.append( mvSlide(p_sympatry, weight=2) )
P_DEC := fnDECCladoProbs(eventProbs=clado_type_probs,
                         eventTypes=clado_event_types,
                         numCharacters=n_areas,
                         maxRangeSize=max_areas)

In [29]:
rf_DEC_tmp <- rep(0, n_states)
rf_DEC_tmp[2] <- 1
rf_DEC <- simplex(rf_DEC_tmp)

In [30]:
m_bg ~ dnPhyloCTMCClado(tree=tree,
                        Q=Q_DEC_epoch,
                        cladoProbs=P_DEC,
                        branchRates=rate_bg,
                        rootFrequencies=rf_DEC,
                        type="NaturalNumbers",
                        nSites=1)

In [51]:
m_bg

   
   NaturalNumbers character matrix with 35 taxa and 1 characters
   Origination:                   ""
   Number of taxa:                35
   Number of included taxa:       35
   Number of characters:          1
   Number of included characters: 1
   Datatype:                      NaturalNumbers
   
   


In [31]:
m_bg.clamp(dat_range_n)

In [32]:
monitors.append( mnScreen(printgen=100, rate_bg, extirpation_rate, distance_scale) )
monitors.append( mnModel(file=out_fn+".model.log", printgen=10) )
monitors.append( mnFile(tree, filename=out_fn+".tre", printgen=10) )
monitors.append( mnJointConditionalAncestralState(tree=tree,
                                                  ctmc=m_bg,
                                                  type="NaturalNumbers",
                                                  withTips=true,
                                                  withStartStates=true,
                                                  filename=out_fn+".states.log",
                                                  printgen=10) )

monitors.append( mnStochasticCharacterMap(ctmc=m_bg,
                                          filename=out_fn+".stoch.log",
                                           printgen=100) )

In [33]:
mymodel = model(m_bg)

In [36]:
mymcmc = mcmc(mymodel, moves, monitors)
mymcmc.run(5000)

   
   Running MCMC simulation
   This simulation runs 1 independent replicate.
   The simulator uses 7 different moves in a random move schedule with 14 moves per iteration
   

Iter        |      Posterior   |     Likelihood   |          Prior   |   distance_s..   |   extirpatio..   |        rate_bg   |    elapsed   |        ETA   |
-------------------------------------------------------------------------------------------------------------------------------------------------------------
0           |       -126.131   |       -126.848   |       0.717109   |           0.01   |         1.0309   |           0.01   |   00:00:00   |   --:--:--   |
100         |        -52.017   |       -46.7471   |       -5.26999   |     0.01483223   |        1.01033   |       4.111559   |   00:00:04   |   --:--:--   |
200         |       -52.3829   |       -47.7938   |       -4.58908   |     0.01073047   |       0.655659   |       2.788017   |   00:00:08   |   00:03:12   |
300         |       -53.1032   

4400        |       -53.1142   |       -48.1106   |       -5.00361   |     0.01414141   |      0.8898458   |       3.709379   |   00:02:55   |   00:00:23   |
4500        |       -53.5991   |       -47.0831   |       -6.51593   |     0.01292004   |       1.558687   |       5.030956   |   00:02:59   |   00:00:19   |
4600        |       -52.9927   |       -47.3376   |       -5.65509   |     0.01390278   |       1.213146   |        4.26327   |   00:03:03   |   00:00:15   |
4700        |       -52.8862   |       -46.9186   |       -5.96763   |     0.01947165   |       1.175016   |       6.253411   |   00:03:07   |   00:00:11   |
4800        |       -53.2725   |       -47.4691   |       -5.80344   |     0.01887137   |       0.942503   |        7.72679   |   00:03:11   |   00:00:07   |
4900        |       -53.5315   |       -47.2504   |       -6.28116   |     0.01703348   |       1.459606   |       4.890691   |   00:03:15   |   00:00:03   |
5000        |       -53.0151   |       -46.7571   | 

In [39]:
%shell figtree output/epoch.tre

javax.swing.UIManager$LookAndFeelInfo[Metal javax.swing.plaf.metal.MetalLookAndFeel]


javax.swing.UIManager$LookAndFeelInfo[Nimbus javax.swing.plaf.nimbus.NimbusLookAndFeel]


javax.swing.UIManager$LookAndFeelInfo[CDE/Motif com.sun.java.swing.plaf.motif.MotifLookAndFeel]


javax.swing.UIManager$LookAndFeelInfo[GTK+ com.sun.java.swing.plaf.gtk.GTKLookAndFeel]


Gtk-Message: 09:51:27.425: Failed to load module "canberra-gtk-module"


